# WT — Statistical analysis and pairwise behavioral heatmaps

**Purpose:** fast analysis using previously cached mouse-level metrics. Run this notebook without downloading Neuropixels spike trains. The time-consuming extraction is handled by `02_process_and_cache.ipynb`.

**Output:** `05_figures/WT/heatmap by zones/<REGION>/PNG/` and `/PDF/`. Filenames start with the electrophysiological metric, e.g. `intraburst_hz_WT_MOs_heatmap.png`.

**Design:** six behavioral states; 15 unique paired comparisons per heatmap (Wilcoxon signed-rank, Holm correction); ≥5 mice with data in **all** states. Mouse is the independent unit. Effect = column minus row. Read `README_HOW_TO_READ_HEATMAPS.md` in the figure directory for metric and brain-region definitions.

**Note:** within-heatmap Holm does not correct across all region×metric heatmaps. These findings are exploratory.

In [ ]:
# ============================================================
# WT — BEHAVIORAL HEATMAPS | INPUTS AND DIRECTORIES
# ============================================================
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from IPython.display import display

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned')
GENOTYPE = 'WT'
INPUT = ROOT / '04_results' / GENOTYPE / 'electrophysiology_v1' / 'WT_mouse_region_phase_metrics.csv'
FIG_ROOT = ROOT / '05_figures' / GENOTYPE / 'heatmap by zones'
TAB_ROOT = ROOT / '04_results' / GENOTYPE / 'electrophysiology_v1' / 'heatmaps_by_region'

assert INPUT.is_file(), f'Missing precomputed mouse-level data: {INPUT}'
df = pd.read_csv(INPUT)
assert {'subject','region','state'}.issubset(df.columns)
FIG_ROOT.mkdir(parents=True, exist_ok=True)
TAB_ROOT.mkdir(parents=True, exist_ok=True)

REGIONS = ['DG','CA1','CA3','SUB','VISp','MOs','CP','MRN','APN']
STATES = ['Q','S','M','F_reward','F_error','I']
STATE_LABELS = ['Quiescence','Stimulus','Movement','Reward','Error','Inter-trial']
METRICS = {
    'fr_hz': ('Firing rate','Hz'),
    'mean_isi_ms': ('Mean ISI','ms'),
    'cv_isi': ('ISI coefficient of variation',''),
    'cv2': ('Local ISI variability (CV2)',''),
    'burst_rate_min': ('Burst rate','bursts/min'),
    'burst_fraction_pct': ('Spikes in bursts','%'),
    'burst_duration_ms': ('Burst duration','ms'),
    'spikes_per_burst': ('Spikes per burst','spikes'),
    'intraburst_hz': ('Intraburst firing rate','Hz'),
    'interburst_interval_s': ('Interburst interval','s'),
}
MIN_MICE = 5
assert set(STATES).issubset(set(df['state'].unique())), 'Some behavioral states are missing'
assert all(metric in df.columns for metric in METRICS), 'Some metric columns are missing'
print(f'Loaded {len(df)} mouse-region-state rows from {INPUT}')
print('WT mice:', df.subject.nunique(), 'regions:', df.region.nunique())
print('New heatmap folder:', FIG_ROOT)

## Generate all WT region × metric heatmaps (and CSV tables)

In [ ]:
# ============================================================
# WT — ALL 90 REGION × METRIC PAIRWISE HEATMAPS
# Each cell: mean(COLUMN - ROW), paired within each mouse.
# For each heatmap: 15 distinct paired Wilcoxon tests + Holm.
# Same complete-case mice for all six states in that heatmap.
# ============================================================

def p_stars(p):
    if not np.isfinite(p): return ''
    if p < .001: return '***'
    if p < .01: return '**'
    if p < .05: return '*'
    return ''

def analyze_six_states(mat, region, metric):
    delta = np.zeros((6,6),dtype=float)
    p_holm_matrix = np.full((6,6), np.nan)
    tests=[]
    for i,j in combinations(range(6),2):
        row_state,col_state=STATES[i],STATES[j]
        a=mat[row_state].to_numpy(dtype=float)
        b=mat[col_state].to_numpy(dtype=float)
        paired_diff=b-a
        if np.all(paired_diff == 0):
            W,p=0.0,1.0
        else:
            test=wilcoxon(a,b,alternative='two-sided',zero_method='wilcox',method='auto')
            W,p=float(test.statistic),float(test.pvalue)
        change=float(paired_diff.mean())
        delta[i,j]=change
        delta[j,i]=-change
        tests.append(dict(region=region,metric=metric,row_state=row_state,
                          column_state=col_state,n_mice=len(mat),
                          mean_row=float(a.mean()),mean_column=float(b.mean()),
                          mean_difference_col_minus_row=change,
                          median_paired_difference=float(np.median(paired_diff)),
                          wilcoxon_W=W,p_raw=p,i=i,j=j))
    corrected=multipletests([t['p_raw'] for t in tests], method='holm')[1]
    for t,pc in zip(tests,corrected):
        t['p_holm']=float(pc)
        t['significant_holm']=bool(pc < .05)
        i,j=t['i'],t['j']
        p_holm_matrix[i,j]=p_holm_matrix[j,i]=pc
        del t['i'],t['j']
    return delta,p_holm_matrix,pd.DataFrame(tests)

def save_heatmap(delta,p_mat,region,metric,unit,n_mice):
    region_dir=FIG_ROOT/region
    png_dir=region_dir/'PNG'
    pdf_dir=region_dir/'PDF'
    png_dir.mkdir(parents=True,exist_ok=True)
    pdf_dir.mkdir(parents=True,exist_ok=True)
    basename=f'{metric}_WT_{region}_heatmap'
    finite=delta[np.isfinite(delta)]
    peak=float(np.max(np.abs(finite))) if finite.size else 1.0
    if not np.isfinite(peak) or peak==0:peak=1.0
    fig,ax=plt.subplots(figsize=(10.5,8.5))
    im=ax.imshow(delta,cmap='RdBu_r',norm=TwoSlopeNorm(vmin=-peak,vcenter=0,vmax=peak))
    for i in range(6):
        for j in range(6):
            d=delta[i,j]
            label='—' if i==j else f'{d:+.2f}{p_stars(p_mat[i,j])}'
            color='white' if abs(d)>0.65*peak else '#202020'
            ax.text(j,i,label,ha='center',va='center',fontsize=11,fontweight='bold',color=color)
    ax.set_xticks(range(6),STATE_LABELS,rotation=35,ha='right')
    ax.set_yticks(range(6),STATE_LABELS)
    ax.set_xlabel('Column state')
    ax.set_ylabel('Row state')
    ax.set_xticks(np.arange(-.5,6,1),minor=True)
    ax.set_yticks(np.arange(-.5,6,1),minor=True)
    ax.grid(which='minor',color='white',linewidth=2)
    ax.tick_params(which='minor',bottom=False,left=False)
    fig.colorbar(im,ax=ax,fraction=.045,pad=.04).set_label(
        f'Column − Row ({unit})' if unit else 'Column − Row')
    ax.set_title(f'WT | {region} | {METRICS[metric][0]}\nPaired behavioral-state differences (n={n_mice} mice)',
                 fontsize=15,fontweight='bold',pad=18)
    fig.subplots_adjust(left=.19,right=.87,bottom=.23,top=.84)
    fig.text(.5,.035,'Positive = higher in column state; negative = lower.\n'
             '* p<.05   ** p<.01   *** p<.001 (paired Wilcoxon; Holm across 15 pairs)',
             ha='center',fontsize=10)
    fig.savefig(png_dir/f'{basename}.png',dpi=350,bbox_inches='tight',facecolor='white')
    fig.savefig(pdf_dir/f'{basename}.pdf',bbox_inches='tight',facecolor='white')
    plt.close(fig)

all_tests=[]
overview=[]
for region in REGIONS:
    sub=df.loc[df.region.eq(region)]
    stats_folder=TAB_ROOT/region
    stats_folder.mkdir(parents=True,exist_ok=True)
    for metric,(title,unit) in METRICS.items():
        mat=sub.pivot_table(index='subject',columns='state',values=metric,aggfunc='mean')
        mat=mat.reindex(columns=STATES).dropna()
        n=len(mat)
        if n < MIN_MICE:
            overview.append(dict(region=region,metric=metric,n_mice=n,status='insufficient_data',n_significant_pairs=0))
            print(f'SKIP {region:5s} {metric:25s}: {n} complete mice')
            continue
        delta,p_adj,tests=analyze_six_states(mat,region,metric)
        tests.to_csv(stats_folder/f'{metric}_WT_{region}_pairwise.csv',index=False)
        save_heatmap(delta,p_adj,region,metric,unit,n)
        all_tests.append(tests)
        overview.append(dict(region=region,metric=metric,n_mice=n,
                             status='completed',n_significant_pairs=int(tests.significant_holm.sum())))
    print(f'Completed region: {region}')

summary=pd.DataFrame(overview)
summary.to_csv(TAB_ROOT/'WT_heatmap_overview.csv',index=False)
if all_tests:
    combined=pd.concat(all_tests,ignore_index=True)
    combined.to_csv(TAB_ROOT/'WT_all_regions_pairwise_statistics.csv',index=False)
    combined.loc[combined.significant_holm].sort_values('p_holm').to_csv(
        TAB_ROOT/'WT_significant_pairwise_comparisons.csv',index=False)
else:
    combined=pd.DataFrame()
print('Generated heatmaps:',int((summary.status=='completed').sum()))
print('Significant comparisons:',int(combined.significant_holm.sum()) if len(combined) else 0)
display(summary.pivot(index='region',columns='metric',values='n_significant_pairs').reindex(index=REGIONS,columns=list(METRICS)))
print('PNG/PDF output:',FIG_ROOT)
print('Statistics output:',TAB_ROOT)